In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import pickle

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Device: cuda


In [2]:
text = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt").text
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.block_size = block_size
        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x).view(B, T, self.n_head, self.head_size).transpose(1, 2)
        q = self.query(x).view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_head, self.head_size).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=96, n_head=6, n_layer=4, block_size=64):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            return logits
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

In [4]:
# Training (short for today)
block_size = 64
batch_size = 64

def get_batch(split):
    n = int(0.9 * len(data))
    d = data[:n] if split == 'train' else data[n:]
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

model = MiniGPT(vocab_size, n_embd=96, n_head=6, n_layer=4, block_size=block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(3000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 600 == 0:
        print(f'Epoch {epoch:4d} | Loss: {loss.item():.4f}')

Epoch    0 | Loss: 4.3563
Epoch  600 | Loss: 1.9032
Epoch 1200 | Loss: 1.6631
Epoch 1800 | Loss: 1.5400
Epoch 2400 | Loss: 1.4634


In [5]:
torch.save(model.state_dict(), 'minigpt_model.pth')
print("Model saved!")

# Load example
# model.load_state_dict(torch.load('minigpt_model.pth'))
# model.to(device)

Model saved!


In [6]:
#  Perplexity Evaluation
@torch.no_grad()
def evaluate_perplexity(model, data, block_size=64, batch_size=32):
    model.eval()
    total_loss = 0
    n_batches = 50
    for _ in range(n_batches):
        xb, yb = get_batch('val')
        logits, loss = model(xb, yb)
        total_loss += loss.item()
    avg_loss = total_loss / n_batches
    perplexity = torch.exp(torch.tensor(avg_loss))
    print(f'Validation Perplexity: {perplexity.item():.2f}')
    return perplexity

evaluate_perplexity(model, data)

Validation Perplexity: 5.19


tensor(5.1936)

In [8]:
#  Advanced Generation
@torch.no_grad()
def generate(model, start_text="ROMEO:", max_new_tokens=400, temperature=0.9, top_k=40):
    model.eval()
    context = torch.tensor(encode(start_text), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        context_crop = context[:, -model.block_size:]
        logits = model(context_crop)
        logits = logits[:, -1, :] / temperature

        # Top-k sampling
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        context = torch.cat((context, next_token), dim=1)

    return decode(context[0].tolist())

print("=== Generated Text ===")
print(generate(model, start_text="JULIET:", temperature=0.7, top_k=50))

=== Generated Text ===
JULIET:
Ay, then to the people, my lord, I may gaze
With woman, chasious that on father deserve
Too knops but at out this consent chase:
You have a hand fain our forth that nant this is
almost up, and the carefing.

KING HENRY VI:
I beser, that, sir, and hope die thee!

First Senatious Romeo,
Which is with thee readies thy seat both,
And yours; that were shall this noble and by a
peace, the shame of all 
